In [1]:
s="The cat is sleepy because it did not sleep last night"

In [2]:
tokens=s.split()

In [3]:
print(tokens)

['The', 'cat', 'is', 'sleepy', 'because', 'it', 'did', 'not', 'sleep', 'last', 'night']


In [5]:
import torch
embedding = torch.nn.Embedding(11, 4)#for learning purpose random embediing
print(embedding.weight)


Parameter containing:
tensor([[ 0.5863,  0.5164, -0.1905, -0.1138],
        [-0.1520,  0.8007, -0.2907, -0.2667],
        [-0.1577, -0.7128, -0.0982,  0.4300],
        [ 1.4444, -0.0420, -0.7851,  0.4064],
        [-0.3595,  0.8646, -0.6008, -0.2718],
        [-0.6428,  1.3397,  1.0024,  0.9274],
        [ 0.0160,  0.9400,  0.7519,  1.3426],
        [ 1.6069,  1.8095,  0.0346, -1.1222],
        [-0.0732, -0.3503,  0.6864, -0.4737],
        [-0.9005,  0.3709, -1.2260, -0.6516],
        [-0.6540, -0.3345, -1.1272,  0.4166]], requires_grad=True)


In [9]:
d_model = embedding.weight.shape[1]   # embedding dimension
d_k = 4                # attention dimension

W_Q = torch.randn(d_model, d_k)
W_K = torch.randn(d_model, d_k)
W_V = torch.randn(d_model, d_k)

print(W_Q)
print(W_K)
print(W_V)

tensor([[ 0.3223, -1.4457,  0.6572,  1.4665],
        [-0.0752, -1.4219, -0.9564, -0.7898],
        [ 0.3493,  0.3268, -1.6663,  0.6038],
        [ 0.4376, -1.6159,  0.9889,  0.8960]])
tensor([[-0.7681,  1.0117, -0.1788,  0.4807],
        [-0.0139,  0.2305, -0.5924,  1.4859],
        [ 0.6960,  0.1936, -0.0693,  0.1860],
        [ 1.0321, -0.4327,  0.5072,  0.7534]])
tensor([[-1.0811, -0.1185, -0.3694, -0.2670],
        [-0.6337, -0.5487,  0.6193, -0.3332],
        [-0.3904, -0.7248,  0.4787, -0.7776],
        [-1.6071,  0.4328,  0.3646, -0.5377]])


We need to build vocabulary and assign each unique token a token id , for the sentence that i have chosen it is not relevant , but in a scenario that has repeated words , we would want only one row from embedding matrix to represent one unique word , and hence the need for token id. Then x=embediing(token_id)selects only those rows that reperesent the vocab.

The embedding table stores one vector per vocabulary word.
Token IDs tell the embedding layer which vocabulary word each token is.

In [13]:
vocab = {word: i for i, word in enumerate(tokens)}
token_ids = torch.tensor([vocab[word] for word in tokens])

In [14]:
x=embedding(token_ids)

In [15]:
#now we calculate Q, K ,v
Q=x@W_Q
K=x@W_K
V=x@W_V

In [18]:
#now we calculate q.k transpose
scores = Q @ K.T
print(scores.shape)

torch.Size([11, 11])


In [19]:
d_k =K.shape[1]
scores=scores/torch.sqrt(torch.tensor(d_k))
#we normalize


In [20]:
#softmax , apply row wise
import torch.nn.functional as F
attention_weights=F.softmax(scores,dim=1)
print(attention_weights.shape)
print(attention_weights)

torch.Size([11, 11])
tensor([[4.8313e-02, 7.6229e-02, 1.0578e-01, 3.6036e-02, 9.0970e-02, 1.5704e-01,
         1.2334e-01, 1.4229e-02, 6.0537e-02, 1.2179e-01, 1.6575e-01],
        [5.6490e-02, 6.9456e-02, 1.4140e-01, 5.1155e-02, 7.8330e-02, 1.7166e-02,
         1.4548e-02, 2.4114e-02, 1.5295e-01, 2.2502e-01, 1.6938e-01],
        [8.6560e-02, 6.9376e-02, 7.4826e-02, 1.1663e-01, 6.3962e-02, 1.4809e-01,
         1.9373e-01, 8.9199e-02, 5.7052e-02, 3.8416e-02, 6.2159e-02],
        [5.7829e-03, 1.2662e-02, 3.5055e-02, 4.8634e-03, 1.7391e-02, 4.3204e-01,
         3.9167e-01, 2.2662e-04, 4.3390e-03, 1.3303e-02, 8.2665e-02],
        [4.2323e-02, 4.8710e-02, 1.7028e-01, 4.6080e-02, 5.5597e-02, 6.0595e-03,
         5.6756e-03, 1.1680e-02, 1.7593e-01, 2.4317e-01, 1.9451e-01],
        [3.6993e-02, 9.1579e-02, 7.6352e-02, 1.2300e-02, 1.1807e-01, 1.3872e-01,
         5.9296e-02, 1.3791e-02, 7.8925e-02, 2.3117e-01, 1.4281e-01],
        [1.0038e-02, 3.2860e-02, 3.5040e-02, 3.2662e-03, 4.6745e-02, 4.96

In [21]:
#Output of attention layer
output = attention_weights @ V
print(output.shape)
print(output)

torch.Size([11, 4])
tensor([[-0.3184,  0.0463,  0.3722, -0.0733],
        [ 0.5555,  0.2648, -0.1043,  0.4504],
        [-1.0536, -0.2542,  0.4585, -0.4422],
        [-1.9922, -0.4869,  1.3354, -1.1983],
        [ 0.6984,  0.3597, -0.1704,  0.5264],
        [ 0.2326,  0.0747,  0.2647,  0.2013],
        [-1.4804, -0.4558,  1.2146, -0.9444],
        [ 0.3080,  0.1671,  0.2472,  0.2827],
        [-0.9715, -1.2415,  0.1367, -0.3651],
        [ 0.4370,  0.2010, -0.2134,  0.3278],
        [ 0.1692,  0.4257, -0.1914,  0.3205]], grad_fn=<MmBackward0>)
